In [33]:
import numpy as np

In [34]:
NORTH_IDX = 0
SOUTH_IDX = 1
WEST_IDX = 2
EAST_IDX = 3
LOCAL_IDX = 4
direction_num = 5

mesh_dim_x = 10
mesh_dim_y = 10
tile_num = mesh_dim_x * mesh_dim_y

In [35]:
class Coord:
    def __init__(self, id=0):
        self.x = id % mesh_dim_x
        self.y = id // mesh_dim_y

    def __repr__(self):
        return f"Coordinate(x={self.x}, y={self.y})"

    def is_perimeter(self):
        return (self.x == 0) or (self.x == mesh_dim_x - 1) or (self.y == 0) or (self.y == mesh_dim_y - 1)

    def is_vertical(self):
        return (self.x == 0) or (self.x == mesh_dim_x - 1)

    def is_horizontal(self):
        return self.is_perimeter() and not self.is_vertical()

    def id(self):
        return self.y * mesh_dim_y + self.x

In [36]:
# Returns next tile for given current and destination ones
# Algorith DOR
def next_tile_gen_yx(current_tile, dst_tile):
    next = Coord(current_tile.id())
    if current_tile.y < dst_tile.y:
        next.y += 1
    elif current_tile.y > dst_tile.y:
        next.y -= 1
    elif current_tile.x < dst_tile.x:
        next.x += 1
    elif current_tile.x > dst_tile.x:
        next.x -= 1
    return next

def next_tile_gen_xy(current_tile, dst_tile):
    next = Coord(current_tile.id())
    if current_tile.x < dst_tile.x:
        next.x += 1
    elif current_tile.x > dst_tile.x:
        next.x -= 1
    elif current_tile.y < dst_tile.y:
        next.y += 1
    elif current_tile.y > dst_tile.y:
        next.y -= 1
    return next

In [37]:
# Returns directions current_tile output ID and next_tile input ID
def encode_dirs(current_tile, next_tile):
    if current_tile.y < next_tile.y:
        return (SOUTH_IDX, NORTH_IDX)
    elif current_tile.y > next_tile.y:
        return (NORTH_IDX, SOUTH_IDX)
    elif current_tile.x < next_tile.x:
        return (EAST_IDX, WEST_IDX)
    elif current_tile.x > next_tile.x:
        return (WEST_IDX, EAST_IDX)
    return (LOCAL_IDX, LOCAL_IDX)

In [38]:
# 1 dim - s - source tile
# 2 dim - d - destination tile
# 3 dim - r - router tile
# 4 dim - i - router input channel
# 5 dim - o - router output channel
# `1` if routing path from `s` to `d` goes through `r` from `i` to `o`
# `0` otherwise
routing_tensor_req_dor = np.zeros(
    (tile_num, tile_num, tile_num, direction_num, direction_num))

for s in range(tile_num):
    for d in range(tile_num):
        s_coord = Coord(s)
        d_coord = Coord(d)
        if not (s_coord.is_perimeter() and not d_coord.is_perimeter()):
            continue
        current_tile = s_coord
        i_cur_dir = LOCAL_IDX
        while current_tile.id() != d:
            next_tile = next_tile_gen_yx(current_tile, d_coord)
            (o_cur_dir, i_nxt_dir) = encode_dirs(current_tile, next_tile)
            routing_tensor_req_dor[s][d][current_tile.id()][i_cur_dir][o_cur_dir] = 1
            current_tile = Coord(next_tile.id())
            i_cur_dir = i_nxt_dir
        routing_tensor_req_dor[s][d][d][i_cur_dir][LOCAL_IDX] = 1

print("Routing tensor done")

Routing tensor done


In [39]:
# 1 dim - s - source tile
# 2 dim - d - destination tile
# 3 dim - r - router tile
# 4 dim - i - router input channel
# 5 dim - o - router output channel
# `1` if routing path from `s` to `d` goes through `r` from `i` to `o`
# `0` otherwise
routing_tensor_resp_dor = np.zeros(
    (tile_num, tile_num, tile_num, direction_num, direction_num))

for s in range(tile_num):
    for d in range(tile_num):
        s_coord = Coord(s)
        d_coord = Coord(d)
        if not (not s_coord.is_perimeter() and d_coord.is_perimeter()):
            continue
        current_tile = s_coord
        i_cur_dir = LOCAL_IDX
        while current_tile.id() != d:
            next_tile = next_tile_gen_xy(current_tile, d_coord)
            (o_cur_dir, i_nxt_dir) = encode_dirs(current_tile, next_tile)
            routing_tensor_resp_dor[s][d][current_tile.id()][i_cur_dir][o_cur_dir] = 1
            current_tile = Coord(next_tile.id())
            i_cur_dir = i_nxt_dir
        routing_tensor_resp_dor[s][d][d][i_cur_dir][LOCAL_IDX] = 1

print("Routing tensor done")

Routing tensor done


In [40]:
# 1 dim - s - source tile
# 2 dim - d - destination tile
traffic_matrix_req = np.zeros((tile_num, tile_num))

for s in range(tile_num):
    for d in range(tile_num):
        s_coord = Coord(s)
        d_coord = Coord(d)
        if not (s_coord.is_perimeter() and not d_coord.is_perimeter()):
            continue
        traffic_matrix_req[s][d] = 1.

print("Traffic matrix done")

Traffic matrix done


In [41]:
# 1 dim - s - source tile
# 2 dim - d - destination tile
traffic_matrix_resp = np.zeros((tile_num, tile_num))

for s in range(tile_num):
    for d in range(tile_num):
        s_coord = Coord(s)
        d_coord = Coord(d)
        if not (not s_coord.is_perimeter() and d_coord.is_perimeter()):
            continue
        traffic_matrix_resp[s][d] = 1.

print("Traffic matrix done")

Traffic matrix done


In [42]:
# 1 dim - r - router tile
# 2 dim - i - router input channel
# 3 dim - o - router output channel
router_req_payload_tensor = np.zeros((tile_num, direction_num, direction_num))
router_resp_payload_tensor = np.zeros((tile_num, direction_num, direction_num))

for s in range(tile_num):
    for d in range(tile_num):
        for r in range(tile_num):
            for i in range(direction_num):
                for o in range(direction_num):
                    router_req_payload_tensor[r][i][o] += traffic_matrix_req[s][d] * \
                        routing_tensor_req_dor[s][d][r][i][o]
                    router_resp_payload_tensor[r][i][o] += traffic_matrix_resp[s][d] * \
                        routing_tensor_resp_dor[s][d][r][i][o]

print("Router payload tensor done")

Router payload tensor done


In [43]:
def calc_router_input_payload_tensor(router_payload_tensor):

    # 1 dim - r - router tile
    # 2 dim - i - router input channel
    router_input_payload_tensor = np.zeros((tile_num, direction_num))

    for r in range(tile_num):
        for i in range(direction_num):
            router_input_payload_tensor[r][i] += np.sum(
                router_payload_tensor[r][i])

    return router_input_payload_tensor

In [44]:
def calc_blocking_possibility_internal(request_possibility, i, o, other_num):
    """
    Compute d(i, j, c) for the current F.
    F is a 5x5 array (F[i][j] for i,j=0..4).
    i, j are the main indices in 0..4.
    c is in 0..4.
    """
    # The four other indices besides i
    others = [x for x in range(5) if x != i]

    total = 0.0
    # Loop over all 16 combinations of (k1, k2, k3, k4) in {0,1}^4
    for mask in range(16):  # from 0 to 15
        # Count how many bits are 1
        # and build the product F[...]^(k_t) * (1-F[...])^(1-k_t)
        ksum = 0
        prob_product = 1.0
        for bit_idx in range(4):
            # k_t is either 0 or 1
            k_t = (mask >> bit_idx) & 1
            p = request_possibility[others[bit_idx]][o]
            if k_t == 1:
                prob_product *= p
                ksum += 1
            else:
                prob_product *= (1 - p)

        # If k_1 + k_2 + k_3 + k_4 = c, add to sum
        if ksum == other_num:
            total += prob_product
    return total


def calc_blocking_possibility(request_possibility, i, o):
    blocking_possibility = 0.0
    for other_num in range(1, 5):
        blocking_possibility_internal = calc_blocking_possibility_internal(
            request_possibility, i, o, other_num)
        blocking_possibility += blocking_possibility_internal * \
            (other_num / (other_num + 1))
    return blocking_possibility


def solve_request_possibility(payload_tensor, max_iter=1000, tol=1e-8):
    """
    Solve for F using the fixed-point iteration:
       F[i][j] = a[i][j] / (1 - D[i][j]),
    with D[i][j] computed via the sums over the 4 other indices.

    :param a: known 5x5 numpy array
    :param max_iter: maximum number of iterations
    :param tol: convergence tolerance
    :return: F as a 5x5 numpy array
    """
    # 1) Initialize F (for example, uniform 0.5)
    request_possibility = payload_tensor.copy()

    for it in range(max_iter):
        request_possibility_old = request_possibility.copy()

        for i in range(5):
            for o in range(5):
                blocking_possibility = calc_blocking_possibility(
                    request_possibility, i, o)

                denom = 1.0 - blocking_possibility
                if abs(denom) < 1e-14:
                    # Avoid dividing by zero.
                    # Could set F[i][j] to some fallback value.
                    request_possibility[i][o] = 0.999999 if denom < 0 else 0.0
                else:
                    request_possibility[i][o] = payload_tensor[i][o] / denom

        # Check for convergence
        diff = np.linalg.norm(request_possibility - request_possibility_old)
        if diff < tol:
            break

    # print(f"Finished in {it+1} iterations with diff={diff}")
    return request_possibility

In [45]:
def calc_blocking_possibility_tensor(router_payload_tensor):
    blocking_possibility = np.zeros((tile_num, direction_num, direction_num))

    for r in range(tile_num):
        for i in range(direction_num):
            for o in range(direction_num):
                request_possibility_solution = solve_request_possibility(
                    router_payload_tensor[r], max_iter=500, tol=1e-10)
                blocking_possibility[r][i][o] = calc_blocking_possibility(
                    request_possibility_solution, i, o)

    print("Blocking possibility done")
    return blocking_possibility

In [46]:
def calc_blocking_time(blocking_possibility):
    blocking_time = blocking_possibility.copy()

    for r in range(tile_num):
        for i in range(direction_num):
            for o in range(direction_num):
                blocking_time[r][i][o] /= (1 - blocking_time[r][i][o])

    print("Blocking time done")
    return blocking_time

In [47]:
def calc_request_handle_time(blocking_time):
    request_handle_time = blocking_time.copy()

    for r in range(tile_num):
        for i in range(direction_num):
            for o in range(direction_num):
                request_handle_time[r][i][o] += 1.

    print("Request handle time done")
    return request_handle_time

In [48]:
def calc_mean_input_handle_time(router_input_payload_tensor, router_payload_tensor, request_handle_time):
    mean_input_handle_time = np.zeros((tile_num, direction_num))

    for r in range(tile_num):
        for i in range(direction_num):
            sum_input_handle_time = 0.
            for o in range(direction_num):
                sum_input_handle_time += router_payload_tensor[r][i][o] * \
                    request_handle_time[r][i][o]
            mean_input_handle_time[r][i] = 0. if (abs(sum_input_handle_time) < 1e-9) else sum_input_handle_time / \
                router_input_payload_tensor[r][i]

    print("Mean request handle time done")
    return mean_input_handle_time

In [49]:
def mm1n_queue(lambda_arrival, E_T, N):
    utilization = lambda_arrival * E_T
    pk_values = [(1 - utilization) * utilization**k / (1 - utilization**(N+1))
                 for k in range(N+1)]
    mean_depth = sum((k - 1) * pk_values[k] for k in range(1, N+1))
    return mean_depth

In [50]:
def calc_mean_input_queue_depth(router_input_payload_tensor, mean_input_handle_time):
    mean_input_queue_depth = np.zeros((tile_num, direction_num))

    for r in range(tile_num):
        for i in range(direction_num):
            mean_input_queue_depth[r][i] = mm1n_queue(
                router_input_payload_tensor[r][i], mean_input_handle_time[r][i], 2)
    print("Mean queue depth done")
    return mean_input_queue_depth

In [51]:
def calc_average_latency(routing_tensor, mean_input_queue_depth, mean_input_handle_time):
    average_latency = 0.0
    for r in range(tile_num):
        for i in range(direction_num):
            if abs(routing_tensor[r][i].sum()) > 1e-9:
                average_latency += (mean_input_queue_depth[r][i] + 1.) * \
                    mean_input_handle_time[r][i] + 1.
    return average_latency

In [52]:
def calc_average_latency_tensor(routing_tensor, mean_input_queue_depth, mean_input_handle_time):
    average_latency = np.zeros((tile_num, tile_num))
    print(count)

    for s in range(tile_num):
        for d in range(tile_num):
            if abs(routing_tensor[s][d].sum()) > 1e-9:
                average_latency[s][d] = calc_average_latency(
                    routing_tensor[s][d], mean_input_queue_depth, mean_input_handle_time)

                # print(Coord(s), Coord(d), average_latency[s][d])

    print("Average latency done")
    return average_latency

In [56]:
def calc_average_latency_pir_pipeline(pir):
  pir_per_mem_tile = pir / (mesh_dim_x - 2) / (mesh_dim_y - 2)
  router_req_payload_tensor_copy = router_req_payload_tensor.copy() * pir_per_mem_tile
  router_resp_payload_tensor_copy = router_resp_payload_tensor.copy() * pir_per_mem_tile
  router_req_input_payload_tensor = calc_router_input_payload_tensor(router_req_payload_tensor_copy)
  router_resp_input_payload_tensor = calc_router_input_payload_tensor(router_resp_payload_tensor_copy)
  req_blocking_possibility = calc_blocking_possibility_tensor(router_req_payload_tensor_copy)
  resp_blocking_possibility = calc_blocking_possibility_tensor(router_resp_payload_tensor_copy)
  req_blocking_time = calc_blocking_time(req_blocking_possibility)
  resp_blocking_time = calc_blocking_time(resp_blocking_possibility)
  req_handle_time = calc_request_handle_time(req_blocking_time)
  resp_handle_time = calc_request_handle_time(resp_blocking_time)
  req_mean_input_handle_time = calc_mean_input_handle_time(router_req_input_payload_tensor, router_req_payload_tensor_copy, req_handle_time)
  resp_mean_input_handle_time = calc_mean_input_handle_time(router_resp_input_payload_tensor, router_resp_payload_tensor_copy, resp_handle_time)
  req_mean_input_queue_depth = calc_mean_input_queue_depth(router_req_input_payload_tensor, req_mean_input_handle_time)
  resp_mean_input_queue_depth = calc_mean_input_queue_depth(router_resp_input_payload_tensor, resp_mean_input_handle_time)
  req_average_latency = calc_average_latency_tensor(routing_tensor_req_dor, req_mean_input_queue_depth, req_mean_input_handle_time)
  resp_average_latency = calc_average_latency_tensor(routing_tensor_resp_dor, resp_mean_input_queue_depth, resp_mean_input_handle_time)
  return (req_average_latency.sum(), resp_average_latency.sum())

In [60]:
pir_list = [0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4]
# pir_list = [0.05, 0.1]
latencies_req = list()
latencies_resp = list()
for pir in pir_list:
  (req_lat, resp_lat) = calc_average_latency_pir_pipeline(pir)
  latencies_req.append(req_lat / 2 / (mesh_dim_x + mesh_dim_y - 2) / (mesh_dim_x - 2) / (mesh_dim_y - 2))
  latencies_resp.append(resp_lat / 2 / (mesh_dim_x + mesh_dim_y - 2) / (mesh_dim_x - 2) / (mesh_dim_y - 2))
  print(f"{pir} done: latency_req = {latencies_req[-1]}; latency_resp = {latencies_resp[-1]}")

Blocking possibility done
Blocking possibility done
Blocking time done
Blocking time done
Request handle time done
Request handle time done
Mean request handle time done
Mean request handle time done
Mean queue depth done
Mean queue depth done
Average latency done
Average latency done
0.05 done: latency_req = 16.768510526074685; latency_resp = 16.778096862877888
Blocking possibility done
Blocking possibility done
Blocking time done
Blocking time done
Request handle time done
Request handle time done
Mean request handle time done
Mean request handle time done
Mean queue depth done
Mean queue depth done
Average latency done
Average latency done
0.1 done: latency_req = 16.936371224179947; latency_resp = 16.953395871347094
Blocking possibility done
Blocking possibility done
Blocking time done
Blocking time done
Request handle time done
Request handle time done
Mean request handle time done
Mean request handle time done
Mean queue depth done
Mean queue depth done
Average latency done
Averag

In [61]:
print(latencies_req)
print(latencies_resp)

[16.768510526074685, 16.936371224179947, 17.16001148970548, 17.432003365296048, 17.74885265634639, 18.112628269595728, 18.535696967263764, 19.06325416440207]
[16.778096862877888, 16.953395871347094, 17.183234430396162, 17.461133643068344, 17.784534551559904, 18.15643077791522, 18.590238024337435, 19.133081893660727]
